In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import jax
import jax.numpy as jnp
import jax.nn as jnn


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =================================================================
                                
# =================================================================
class FourierFeatureMapping(nn.Module):
    def __init__(self, in_dims=2, mapping_size=128, scale=10.0):
        super().__init__()
                                  
        self.register_buffer('B', torch.randn(in_dims, mapping_size) * scale)

    def forward(self, x):
        # x: (Batch, 2)
        # x @ B: (Batch, mapping_size)
        x_proj = (2.0 * np.pi * x) @ self.B
                                          
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

# =================================================================
               
# =================================================================
class DenseResBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.norm1 = nn.LayerNorm(size)
        self.norm2 = nn.LayerNorm(size)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = F.silu(self.fc1(x))
        x = self.norm2(x)
        x = F.silu(self.fc2(x))
        return x + res

# =================================================================
                       
# =================================================================
class HighFidelityNeuralVF(nn.Module):
    def __init__(self, n_poles=8, width=256, fourier_size=128):
        super().__init__()
        self.n_poles = n_poles
        
                                         
        self.fourier_map = FourierFeatureMapping(in_dims=2, mapping_size=fourier_size, scale=5.0)
        
                                 
        in_features = fourier_size * 2 + 2 
        self.in_layer = nn.Linear(in_features, width)
        
                           
        self.res_layers = nn.ModuleList([DenseResBlock(width) for _ in range(6)])
        
                
        self.out_layer = nn.Linear(width, 2 * n_poles)

    def forward(self, x):
        # x shape: (Batch, 2)
        
                
        x_fourier = self.fourier_map(x)
                                            
        x_concat = torch.cat([x, x_fourier], dim=-1)
        
        h = F.silu(self.in_layer(x_concat))
        for layer in self.res_layers:
            h = layer(h)
        out = self.out_layer(h)
        
                             
        log_tau_k = out[:, 0 : self.n_poles]
        log_w_k = out[:, self.n_poles :]
        
        tau_k = 10.0 ** log_tau_k
        w_k = 10.0 ** log_w_k
                                  
                                        
        w_k_sum = torch.sum(w_k, dim=1, keepdim=True) + 1e-12        
        w_k = w_k / w_k_sum
        
        return tau_k, w_k, log_tau_k, log_w_k

In [ ]:

# =================================================================
              
# =================================================================

N_POLES = 12
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_loaded = np.load('trf_train_data_saved.npy', allow_pickle=True).item()
X_np = data_loaded['X_train']
Y_d0_np = data_loaded['Y_d0']
Y_tau_np = data_loaded['Y_tau']
Y_w_np = data_loaded['Y_w']
print("[*] 已加载 trf_train_data_dict.npy 数据。")

# X_np, Y_d0_np, Y_tau_np, Y_w_np = generate_trf_dataset(n_samples=300, n_poles=N_POLES)

                   
dataset = TensorDataset(
    torch.tensor(X_np, dtype=torch.float32).to(device),
    torch.tensor(Y_d0_np, dtype=torch.float32).to(device),
    torch.tensor((Y_tau_np), dtype=torch.float32).to(device),
    torch.tensor(Y_w_np, dtype=torch.float32).to(device)
)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)



In [ ]:
              
model = HighFidelityNeuralVF(N_POLES).to(device)


In [ ]:
                 
                              
                                      
model.load_state_dict(torch.load("model_trained.pth"))
model.eval()
print("[*] 模型已从 model_trained.pth 加载并进入推理模式")




In [ ]:


class PureJAXIPNet:
    """完全使用 jax.numpy 重写的高精度物理神经网络映射器"""
    
    def __init__(self, params_dict, n_poles=8, dtype=jnp.float32):
                                                                 
        self.params = jax.tree.map(lambda x: jnp.asarray(x, dtype=dtype), params_dict)
        self.n_poles = int(n_poles)

        # =========================================================
                                               
        # =========================================================
        @jax.jit
        def _forward_fn(params, x):
            
            def _fourier_map(x_in):
                B = params['fourier_map.B']
                x_proj = (2.0 * jnp.pi * x_in) @ B
                return jnp.concatenate([jnp.sin(x_proj), jnp.cos(x_proj)], axis=-1)

            def _layer_norm(x_in, weight, bias, eps=1e-5):
                mean = jnp.mean(x_in, axis=-1, keepdims=True)
                var = jnp.var(x_in, axis=-1, keepdims=True)
                return (x_in - mean) / jnp.sqrt(var + eps) * weight + bias

            def _res_block(x_in, prefix):
                fc1_w, fc1_b = params[f'{prefix}.fc1.weight'], params[f'{prefix}.fc1.bias']
                fc2_w, fc2_b = params[f'{prefix}.fc2.weight'], params[f'{prefix}.fc2.bias']
                ln1_w, ln1_b = params[f'{prefix}.norm1.weight'], params[f'{prefix}.norm1.bias']
                ln2_w, ln2_b = params[f'{prefix}.norm2.weight'], params[f'{prefix}.norm2.bias']

                res = x_in
                x_tmp = _layer_norm(x_in, ln1_w, ln1_b)
                x_tmp = jnn.silu(x_tmp @ fc1_w.T + fc1_b)
                
                x_tmp = _layer_norm(x_tmp, ln2_w, ln2_b)
                x_tmp = jnn.silu(x_tmp @ fc2_w.T + fc2_b)
                return x_tmp + res

            x_fourier = _fourier_map(x)
            x_concat = jnp.concatenate([x, x_fourier], axis=-1)

            in_w, in_b = params['in_layer.weight'], params['in_layer.bias']
            h = jnn.silu(x_concat @ in_w.T + in_b)

                         
            for i in range(6):
                h = _res_block(h, f'res_layers.{i}')

            out_w, out_b = params['out_layer.weight'], params['out_layer.bias']
            out = h @ out_w.T + out_b

                                                 
            log_tau_k = out[:, 0 : self.n_poles]
            log_w_k = out[:, self.n_poles :]
            
            tau_k = 10.0 ** log_tau_k
            w_k = 10.0 ** log_w_k
                                 
                                            
            w_k_sum = jnp.sum(w_k, axis=1, keepdims=True) + 1e-12        
       
            w_k = w_k / w_k_sum 
       
        
            return  tau_k, w_k

                        
        self._jit_forward = _forward_fn

    def __call__(self, x):
                                                       
        return self._jit_forward(self.params, x)




def load_jax_model(filepath, n_poles=18):
    """
    从硬盘加载权重并重建 PureJAXIPNet 模型
    """
               
    loaded_data = np.load(filepath)
    
                                             
    loaded_params = {key: loaded_data[key] for key in loaded_data.files}
    
                          
    new_model = PureJAXIPNet(loaded_params, n_poles=n_poles)
    print(f"✅ 成功从 {filepath} 加载模型并初始化 JAX 引擎！")
    
    return new_model

      
jax_model = load_jax_model("jax_ip_model_12poles.npz", n_poles=12)


In [ ]:
with torch.no_grad():
                               
    X_np_expanded = np.tile(X_np, (10, 1))  # (N*10, dim)
    with torch.no_grad():
        tau_pred, w_pred, _, _ = model(torch.tensor(X_np_expanded, dtype=torch.float32).to(device))
    

In [ ]:
tau_jax, w_jax,= jax_model(X_np_expanded)

In [ ]:
# =================================================================
                     
# =================================================================
model.eval()
num= 10

test_tau_val = 10**X_np[num,0]
test_c_val = X_np[num,1]
test_tau_val = 0.1
test_c_val = 0.1


In [ ]:

print(f"\n[*] 正在验证测试点 (tau={test_tau_val}, c={test_c_val})...")

                                                
x_test = torch.tensor([[np.log10(test_tau_val), test_c_val]], dtype=torch.float32).to(device)

with torch.no_grad():
    tau_pred, w_pred, _, _ = model(x_test)

tau_jax, w_jax = jax_model(x_test.cpu().numpy())

                        
tau_np = tau_pred.cpu().numpy()[0]
w_np = w_pred.cpu().numpy()[0]

      
freqs = np.logspace(-2, 6, 100)
s = 1j * 2 * np.pi * freqs

    
Y_true = 1.0 / (1.0 + (s * test_tau_val)**test_c_val)
     
Y_pred = np.sum(w_np[:, None] / (1.0 + s[None, :] * tau_np[:, None]), axis=0)


# Y_pred_2 = Y_d0_np[num] + np.sum(Y_w_np[num][:, None] / (1.0 + s[None, :] * Y_tau_np[num][:, None]), axis=0)
                                               
w_jax_squeezed = np.squeeze(w_jax)
tau_jax_squeezed = np.squeeze(tau_jax)

Y_pred_jax =np.sum(
    w_jax_squeezed[:, None] / (1.0 + s[None, :] * tau_jax_squeezed[:, None]), axis=0
)


In [ ]:

plt.figure(figsize=(12, 8))

                  
plt.subplot(2, 2, 1)
plt.semilogx(freqs, Y_true.real, 'k-', lw=3, label="True Cole-Cole (Real)")
plt.semilogx(freqs, Y_pred.real, 'r--', lw=2, label="PyTorch (Real)")
#plt.semilogx(freqs, Y_pred_jax.real, 'b', lw=2, label="Fit (Real)")

plt.title(f"Real Part (tau={test_tau_val}, c={test_c_val})")
plt.legend(); plt.grid(True, alpha=0.3)

                  
plt.subplot(2, 2, 2)
plt.semilogx(freqs, Y_true.imag, 'k-', lw=3, label="True Cole-Cole (Imag)")
plt.semilogx(freqs, Y_pred.imag, 'r--', lw=2, label="PyTorch (Imag)")
#plt.semilogx(freqs, Y_pred_jax.imag, 'b', lw=2, label="Fit (Imag)")
plt.title(f"Imag Part (tau={test_tau_val}, c={test_c_val})")
plt.legend(); plt.grid(True, alpha=0.3)

             
plt.subplot(2, 2, 3)
                 
err_nn_real = 100 * np.abs(Y_true.real - Y_pred.real) / (np.maximum(np.abs(Y_true.real), 1e-12))
err_nn_real2 = 100 * np.abs(Y_true.real - Y_pred_jax.real) / (np.maximum(np.abs(Y_true.real), 1e-12))

plt.loglog(freqs, err_nn_real, 'r-', label="Real Part RelErr")
#plt.loglog(freqs, err_nn_real2, 'b-', label="Real2 Part RelErr")

plt.title("Relative Error (Real) (%)")
plt.axhline(1.0, color='k', ls=':', label='1% Threshold')
plt.legend(); plt.grid(True, which="both", alpha=0.3)

             
plt.subplot(2, 2, 4)
                 
err_nn_imag = 100 * np.abs(Y_true.imag - Y_pred.imag) / np.maximum(np.abs(Y_true.imag), 1e-12)
err_nn_imag2 = 100 * np.abs(Y_true.imag - Y_pred_jax.imag) / np.maximum(np.abs(Y_true.imag), 1e-12)

plt.loglog(freqs, err_nn_imag, 'r-', label="Imag Part RelErr")
#plt.loglog(freqs, err_nn_imag2, 'b-', label="Imag2 Part RelErr")

plt.title("Relative Error (Imag) (%)")
plt.axhline(1.0, color='k', ls=':', label='1% Threshold')
plt.legend(); plt.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
'''# 模型的保存
# 假设你的模型变量名是 model
torch.save(model.state_dict(), "model_trained.pth")
print("[*] 模型已保存为 model_trained.pth")

# 模型的加载
# 重新初始化模型结构，并加载权重
# 假设你的模型类名是 Net（请根据你的实际模型类名替换）
# model = Net()  # 如果未定义Net，则按你的实际模型类名
model.load_state_dict(torch.load("model_trained.pth"))
model.eval()
print("[*] 模型已从 model_trained.pth 加载并进入推理模式")'''